#### Headlamp from Grok

In [ ]:
# 1. Add the official Helm repo
helm repo add headlamp https://kubernetes-sigs.github.io/headlamp/
helm repo update

# 2. Basic install (recommended in kube-system or a dedicated namespace)
# helm install my-headlamp headlamp/headlamp --namespace kube-system --create-namespace

# Example: Install with Ingress + custom host
helm install my-headlamp headlamp/headlamp \
  --namespace headlamp \
  --create-namespace \
  --set ingress.enabled=true \
  --set ingress.hosts[0].host=headlamp.voip.local \
  --set ingress.hosts[0].paths[0].path=/ \
  --set ingress.hosts[0].paths[0].type=Prefix \
	--set ingress.ingressClassName=nginx              

In [ ]:
kubectl rollout status deployment my-headlamp -n headlamp
curl -H "Host: headlamp.voip.local" http://172.16.6.90

In [ ]:
  kubectl create token my-headlamp --namespace headlamp

In [ ]:
# # Force headlamp to same node as ingress
# INGRESS_NODE=$(kubectl get pods -n ingress-nginx -o jsonpath='{.items[0].spec.nodeName}')
# echo "Pin headlamp to: $INGRESS_NODE"

# kubectl patch deployment my-headlamp -n headlamp --patch "{
#   \"spec\": {
#     \"template\": {
#       \"spec\": {
#         \"nodeSelector\": {
#           \"kubernetes.io/hostname\": \"$INGRESS_NODE\"
#         }
#       }
#     }
#   }
# }"

`Helper`

In [ ]:
# Remove from ingress-nginx
kubectl patch deployment ingress-nginx-controller -n ingress-nginx --type=json \
  -p='[{"op":"remove","path":"/spec/template/spec/nodeSelector/ingress-ready"}]'

# Remove from headlamp
kubectl patch deployment my-headlamp -n headlamp --type=json \
  -p='[{"op":"remove","path":"/spec/template/spec/nodeSelector/ingress-ready"}]'

In [ ]:
kubectl edit ippools.crd.projectcalico.org

    ipipMode: Never

In [ ]:
kubectl get all -n headlamp

In [ ]:
kubectl delete pods -n kube-system -l k8s-app=calico-node

In [ ]:
kubectl get ingress -n headlamp

kubectl describe ingress my-headlamp -n headlamp

In [ ]:
# View values / customize
helm show values headlamp/headlamp > my-values.yaml

# Upgrade
helm upgrade my-headlamp headlamp/headlamp -f my-values.yaml -n kube-system

# Uninstall
helm uninstall my-headlamp -n kube-system

helm uninstall my-headlamp -n headlamp

We can use the values file for best practice

In [ ]:
vim values.yaml

In [ ]:
ingress:
  enabled: true
  ingressClassName: nginx          # ← Very important
  annotations:
    nginx.ingress.kubernetes.io/force-ssl-redirect: "true"
    # kubernetes.io/tls-acme: "true"   # if using cert-manager
  hosts:
    - host: headlamp.example.com
      paths:
        - path: /
          pathType: Prefix
  # tls:
  #   - secretName: headlamp-tls
  #     hosts:
  #       - headlamp.example.com

In [ ]:
helm install my-headlamp headlamp/headlamp -f values.yaml --namespace headlamp --create-namespace

#### Nexus